In [5]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import time
import os

In [7]:
def getDateofNewsfromCSV(save_path, symbol):
    if os.path.exists(save_path):
        df = pd.read_csv(save_path)
        print(f"{symbol}news.csv file exists!")

        if not df.empty:
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            df = df.dropna(subset=["Date"])

            if not df.empty:
                save_date = df["Date"].iloc[0]
                return save_date, df

        return None, df

    else:
        print(f"{symbol}news.csv file not exist!")
        return None, None


def fetchFullContent(link, date, headline, data, symbol, i):
    print(f"{i+1} ✓ Fetching {symbol}...", end=" ")

    try:
        response = requests.get(link, timeout=10)
        response.raise_for_status()
        article_soup = BeautifulSoup(response.content, "html.parser")

        content_div = article_soup.find("div", id="newsdetail-content")
        content = (
            content_div.get_text(separator="\n", strip=True)
            if content_div else ""
        )

    except Exception as e:
        print(f"Failed to fetch {link}: {e}")
        content = ""

    data["Date"].append(date)
    data["Headline"].append(headline)
    data["Link"].append(link)
    data["Full Content"].append(content)

    print("✓ Data appended")
    return



query = "NEPSECompanyExtractor"
companyDetails = pd.read_csv(f"../DATA-HTML-STOCK/NEPSECompany/{query}.csv")
symbols = companyDetails["Symbol"]
stock_ignored = ["CORBLP", "GFCLPO", "HBLPO", "LFCPO", "SBIPO", "GABLPO", "AMFIPO", "NMLBS", "ILFCPO"]
stock_no_news = ["HATHPO", "BUDBLP", "CFCLPO", "CORBLP", "EBLPO", "HAMROP", "HGIPO", "JFLPO", "JBNLPO", "KADBLP", "KNBLPO", "CEFLPO", "LBLPO", "MFILPO", "MIDBLP", "NBBLPO", "NABBPO", "NABBPO", "NBBPO", "PFLPO", "PRINPO", "PURBLP", "SBBLJP", "SIFCPO", "SILPO", "SMFDBP", "SYFLPO", "TNBLPO", "TDBLPO", "UFLPO", "WDBLPO", "WMBFPO", "HLBSLP"]
# symbol_s = ["GRDBL", "GMFIL"]
# GRDBL GMFIL

for index, symbol in enumerate(symbols):

    if symbol in stock_ignored:
        print(f"\n{symbol} is in ignore list!\n")
        continue

    data = {"Date": [],"Headline": [],"Link": [],"Full Content": []}

    html_path = f"../DATA-HTML-STOCK/webScrapped-htmlfiles/news-WEB-SCRAP-htmlfile/{symbol}news.html" 
    save_path = f"../DATA-HTML-STOCK/NEPSENEWS/{symbol}news.csv"

    save_date, old_df = getDateofNewsfromCSV(save_path, symbol)

    print(f"\n✓ Parsing HTML {symbol}...")

    if not os.path.exists(html_path):
        print(f"{symbol}news.html file not exist")
        continue

    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    soupbdy = soup.find("tbody")

    if soupbdy is None:
        print(f"No news table found for {symbol}")
        continue

    rows = soupbdy.find_all("tr")

    for i, row in enumerate(rows):

        cols = row.find_all("td")

        if len(cols) < 2:
            continue

        date_text = cols[0].get_text(strip=True)
        parsed_date = pd.to_datetime(date_text, errors="coerce")

        if pd.isna(parsed_date):
            print(f"⚠ Skipping invalid date: {date_text}")
            continue

        date = parsed_date

        headline = cols[1].get_text(strip=True)

        link_tag = cols[1].find("a")
        if not link_tag or not link_tag.get("href"):
            continue

        link = link_tag["href"].strip()

        if save_date is not None:
            if date <= save_date:
                break

        fetchFullContent(link, date, headline, data, symbol, i)
        time.sleep(1)

    new_df = pd.DataFrame(data)

    if save_date is not None:

        if not new_df.empty:
            final_df = pd.concat(
                [new_df, old_df],
                ignore_index=True
            )
            final_df = final_df.sort_values(
                by="Date", ascending=False
            )
            final_df.to_csv(save_path, index=False)
            print(f"✓ Updated {symbol}news.csv")

        else:
            print(f"No new news for {symbol}")

    else:
        if not new_df.empty:
            new_df = new_df.sort_values(
                by="Date", ascending=False
            )
            new_df.to_csv(save_path, index=False)
            print(f"✓ Created {symbol}news.csv")

    print(f"✓ {len(symbols) - index} remain to parse...\n")


print("All Done!")

ADBLnews.csv file exists!

✓ Parsing HTML ADBL...
No new news for ADBL
✓ 300 remain to parse...

APInews.csv file exists!

✓ Parsing HTML API...
1 ✓ Fetching API... ✓ Data appended
✓ Updated APInews.csv
✓ 299 remain to parse...

HATHnews.csv file exists!

✓ Parsing HTML HATH...
No new news for HATH
✓ 298 remain to parse...

HATHPOnews.csv file not exist!

✓ Parsing HTML HATHPO...
HATHPOnews.html file not exist
AKPLnews.csv file exists!

✓ Parsing HTML AKPL...
No new news for AKPL
✓ 296 remain to parse...

AHPCnews.csv file exists!

✓ Parsing HTML AHPC...
No new news for AHPC
✓ 295 remain to parse...

ALICLnews.csv file exists!

✓ Parsing HTML ALICL...
No new news for ALICL
✓ 294 remain to parse...

ALICLPnews.csv file exists!

✓ Parsing HTML ALICLP...
No new news for ALICLP
✓ 293 remain to parse...

BOKLnews.csv file exists!

✓ Parsing HTML BOKL...
No new news for BOKL
✓ 292 remain to parse...

BOKLPOnews.csv file exists!

✓ Parsing HTML BOKLPO...
No new news for BOKLPO
✓ 291 remain to